# Python Lists - Solutions
### A Python Buddy Guide

---

Full solutions for all 10 list practice questions. Each solution includes:
- **Explanatory approach** - clear step-by-step logic
- **Pythonic approach** - concise idiomatic Python
- **Key insight** - what makes this problem tick

---
## Q1 - Rearrange List Ends

In [ ]:
# --- Explanatory ---
def rearrange_v1(lst):
    if len(lst) <= 1:
        return lst[:]
    first  = lst[0]
    last   = lst[-1]
    middle = lst[1:-1]    # everything except first and last
    return [last] + middle + [first]

# --- Pythonic ---
def rearrange_v2(lst):
    if len(lst) <= 1:
        return lst[:]
    return lst[-1:] + lst[1:-1] + lst[:1]

# Tests
for fn in (rearrange_v1, rearrange_v2):
    print(fn([1, 2, 3, 4, 5]))   # [5, 2, 3, 4, 1]
    print(fn([10, 20]))           # [20, 10]
    print(fn([7]))                # [7]
    print()

**Key insight:** `lst[-1:]` gives a one-element list (not just the element), so concatenation with `+` works cleanly. `lst[1:-1]` naturally gives an empty list when `len == 2`, so no special case is needed beyond `len <= 1`.

---
## Q2 - Clean and Summarise a List

In [ ]:
# --- Explanatory ---
def clean_summary_v1(lst):
    cleaned = []
    for x in lst:
        if x != 0:
            cleaned.append(x)
    cleaned.sort()
    if not cleaned:
        return ([], 0, 0.0)
    total   = sum(cleaned)
    average = round(total / len(cleaned), 2)
    return (cleaned, total, average)

# --- Pythonic (list comprehension) ---
def clean_summary_v2(lst):
    cleaned = sorted(x for x in lst if x != 0)
    if not cleaned:
        return ([], 0, 0.0)
    total = sum(cleaned)
    return (cleaned, total, round(total / len(cleaned), 2))

# Tests
for fn in (clean_summary_v1, clean_summary_v2):
    print(fn([3, 0, 1, 0, 4, 2]))    # ([1, 2, 3, 4], 10, 2.5)
    print(fn([0, 0, 0]))              # ([], 0, 0.0)
    print(fn([7, 2, 0, 9, 0, 3]))    # ([2, 3, 7, 9], 21, 5.25)
    print()

**Key insight:** Check for an empty list *after* filtering (before computing average) to avoid a `ZeroDivisionError`. `sorted(generator)` is equivalent to building a list comprehension and then sorting - one clean step.

---
## Q3 - Count Pairs with Target Sum

In [ ]:
# --- Explanatory ---
def count_pairs_v1(lst, target):
    count = 0
    for i in range(len(lst)):
        for j in range(i + 1, len(lst)):   # j always > i, so no duplicates
            if lst[i] + lst[j] == target:
                count += 1
    return count

# --- Pythonic (sum over a generator) ---
def count_pairs_v2(lst, target):
    return sum(
        1
        for i in range(len(lst))
        for j in range(i + 1, len(lst))
        if lst[i] + lst[j] == target
    )

# Tests
for fn in (count_pairs_v1, count_pairs_v2):
    print(fn([1, 2, 3, 4, 3], 6))   # 2
    print(fn([1, 5, 3, 2, 4], 6))   # 2
    print(fn([1, 1, 1, 1], 2))      # 6
    print(fn([1, 2, 3], 10))        # 0
    print()

**Key insight:** Starting the inner loop at `i + 1` (not `0`) automatically guarantees `i < j` and prevents counting the same pair twice. This is the standard nested-loop pair enumeration pattern.

---
## Q4 - Comprehension: Categorise Numbers

In [ ]:
# --- Explanatory ---
def categorise_v1(lst):
    negatives = []
    positives = []
    zero_count = 0
    for x in lst:
        if x < 0:
            negatives.append(x)
        elif x > 0:
            positives.append(x)
        else:
            zero_count += 1
    return {
        "negative": sorted(negatives),
        "zero":     zero_count,
        "positive": sorted(positives),
    }

# --- Pythonic (list comprehensions) ---
def categorise_v2(lst):
    return {
        "negative": sorted(x for x in lst if x < 0),
        "zero":     lst.count(0),
        "positive": sorted(x for x in lst if x > 0),
    }

# Tests
for fn in (categorise_v1, categorise_v2):
    print(fn([-3, 0, 2, -1, 0, 5, 4]))
    # {'negative': [-3, -1], 'zero': 2, 'positive': [2, 4, 5]}
    print(fn([1, 2, 3]))
    # {'negative': [], 'zero': 0, 'positive': [1, 2, 3]}
    print()

**Key insight:** `lst.count(0)` is the cleanest way to count zeros. `sorted(generator)` works directly - no intermediate list needed.

---
## Q5 - Sort Words by Frequency then Alphabetically

In [ ]:
# --- Explanatory ---
def sort_by_frequency_v1(words):
    # Build a frequency dictionary
    freq = {}
    for w in words:
        freq[w] = freq.get(w, 0) + 1
    # Sort: primary key = -frequency (so high freq comes first)
    #        secondary key = word (alphabetical tiebreak)
    return sorted(words, key=lambda w: (-freq[w], w))

# --- Pythonic (using list.count as key) ---
# Works well for short lists; O(n²) but readable
def sort_by_frequency_v2(words):
    return sorted(words, key=lambda w: (-words.count(w), w))

# Tests
for fn in (sort_by_frequency_v1, sort_by_frequency_v2):
    print(fn(["the", "cat", "sat", "the", "cat", "the"]))
    # ['the', 'the', 'the', 'cat', 'cat', 'sat']
    print(fn(["a", "b", "a", "c", "b"]))
    # ['a', 'a', 'b', 'b', 'c']
    print(fn(["apple", "fig", "banana"]))
    # ['apple', 'banana', 'fig']
    print()

**Key insight:** Using a **tuple as the sort key** `(-freq, word)` elegantly handles multiple sort criteria in one `sorted()` call. Negating the frequency flips descending to ascending so Python's default ascending sort works correctly. Python's sort is **stable**, so equal keys preserve relative order - but here the tuple key makes every pair of words fully comparable.

---
## Q6 - Rotate Matrix 90° Clockwise

In [ ]:
# --- Explanatory ---
def rotate_90_v1(matrix):
    n = len(matrix)
    # In a 90° CW rotation: new[col][n-1-row] = old[row][col]
    # Equivalently:          new[i][j]         = old[n-1-j][i]
    result = [[0] * n for _ in range(n)]
    for row in range(n):
        for col in range(n):
            result[col][n - 1 - row] = matrix[row][col]
    return result

# --- Pythonic (transpose then reverse each row) ---
def rotate_90_v2(matrix):
    # Step 1: transpose (zip(*matrix) gives columns as tuples)
    # Step 2: reverse each row → clockwise rotation
    return [list(row[::-1]) for row in zip(*matrix)]

# Tests
for fn in (rotate_90_v1, rotate_90_v2):
    print("3×3:")
    for row in fn([[1,2,3],[4,5,6],[7,8,9]]):
        print(row)
    print("2×2:")
    for row in fn([[1,2],[3,4]]):
        print(row)
    print()

**Key insight:** A 90° clockwise rotation = **transpose** + **reverse each row**. `zip(*matrix)` transposes by unpacking rows into `zip`, giving column-wise tuples. Then `row[::-1]` reverses each one. This is one of the most elegant matrix tricks in Python.

---
## Q7 - Saddle Point in a Matrix

In [ ]:
# --- Explanatory ---
def saddle_points_v1(matrix):
    rows    = len(matrix)
    cols    = len(matrix[0])
    result  = []
    # Precompute column minimums
    col_min = [min(matrix[r][c] for r in range(rows)) for c in range(cols)]
    for r in range(rows):
        row_max = max(matrix[r])   # maximum in this row
        for c in range(cols):
            if matrix[r][c] == row_max and matrix[r][c] == col_min[c]:
                result.append((r, c))
    return result

# --- Pythonic (list comprehension) ---
def saddle_points_v2(matrix):
    rows    = len(matrix)
    cols    = len(matrix[0])
    col_min = [min(matrix[r][c] for r in range(rows)) for c in range(cols)]
    return [
        (r, c)
        for r in range(rows)
        for c in range(cols)
        if matrix[r][c] == max(matrix[r]) == col_min[c]
    ]

# Tests
for fn in (saddle_points_v1, saddle_points_v2):
    print(fn([[1,2,3],[4,5,6],[7,8,9]]))           # [(0,2),(1,2),(2,2)]
    print(fn([[3,1,2],[4,1,0],[2,5,3]]))           # [(1,0)]
    print(fn([[5,5],[5,5]]))                        # [(0,0),(0,1),(1,0),(1,1)]
    print()

**Key insight:** Precompute column minimums once before the main loop - avoids recomputing `min` of each column for every cell. The chained comparison `matrix[r][c] == max(matrix[r]) == col_min[c]` is idiomatic Python for checking two equalities at once.

---
## Q8 - Run-Length Encoding

In [ ]:
# --- Explanatory ---
def run_length_encode_v1(lst):
    if not lst:
        return []
    encoded  = []
    count    = 1
    current  = lst[0]
    for elem in lst[1:]:
        if elem == current:
            count += 1
        else:
            encoded.append((count, current))
            current = elem
            count   = 1
    encoded.append((count, current))   # flush the last run
    return encoded

def run_length_decode_v1(encoded):
    result = []
    for count, elem in encoded:
        result.extend([elem] * count)
    return result

# --- Pythonic ---
def run_length_decode_v2(encoded):
    return [elem for count, elem in encoded for _ in range(count)]

# Tests
enc = run_length_encode_v1
print(enc([1, 1, 2, 3, 3, 3, 2, 2]))   # [(2,1),(1,2),(3,3),(2,2)]
print(enc([5, 5, 5, 5, 5]))             # [(5,5)]
print(enc([1, 2, 3]))                   # [(1,1),(1,2),(1,3)]

for dec in (run_length_decode_v1, run_length_decode_v2):
    print(dec([(2,1),(1,2),(3,3),(2,2)]))          # [1,1,2,3,3,3,2,2]
    original = [4, 4, 4, 2, 2, 1, 3, 3]
    print(dec(enc(original)) == original)          # True

**Key insight:** The "flush after loop" pattern - appending the final accumulated run *after* the loop ends - is the standard structure for all run-detection algorithms. It's easy to forget and a common source of off-by-one bugs. For decoding, `[elem] * count` expands a single element into a list of `count` copies, and `extend` appends all of them cleanly.

---
## Q9 - Spiral Boundary Sum

In [ ]:
# --- Explanatory ---
def spiral_boundary_sum_v1(matrix):
    rows = len(matrix)
    cols = len(matrix[0])

    if rows == 1:
        return sum(matrix[0])
    if cols == 1:
        return sum(row[0] for row in matrix)

    boundary = []
    boundary += matrix[0]                        # top row → left to right
    boundary += [matrix[r][-1] for r in range(1, rows)]     # right col ↓ (skip top)
    boundary += matrix[-1][-2::-1]               # bottom row ← (skip right corner)
    boundary += [matrix[r][0] for r in range(rows-2, 0, -1)]  # left col ↑ (skip corners)
    return sum(boundary)

# --- Pythonic (set of boundary indices) ---
def spiral_boundary_sum_v2(matrix):
    rows = len(matrix)
    cols = len(matrix[0])
    return sum(
        matrix[r][c]
        for r in range(rows)
        for c in range(cols)
        if r == 0 or r == rows - 1 or c == 0 or c == cols - 1
    )

# Tests
for fn in (spiral_boundary_sum_v1, spiral_boundary_sum_v2):
    print(fn([[1,2,3,4],[5,6,7,8],[9,10,11,12]]))   # 65
    print(fn([[1,2,3],[4,5,6],[7,8,9]]))             # 40
    print(fn([[5]]))                                  # 5
    print(fn([[1,2,3]]))                              # 6
    print()

**Key insight:** The Pythonic solution recognises that boundary elements are simply those where `r == 0` or `r == rows-1` or `c == 0` or `c == cols-1` - no traversal order needed for a *sum*. The explicit traversal (v1) would matter if you needed the elements *in order*, but for summing, the index-based filter is far simpler.

---
## Q10 - Tournament Ranking

In [ ]:
# --- Explanatory ---
def rank_players_v1(players, results):
    # Initialise stats for each player
    wins   = {p: 0 for p in players}
    losses = {p: 0 for p in players}

    for winner, loser in results:
        wins[winner]   += 1
        losses[loser]  += 1

    # Points = 3 per win
    points = {p: wins[p] * 3 for p in players}

    # Sort: descending points → ascending losses → alphabetical name
    return sorted(players, key=lambda p: (-points[p], losses[p], p))

# --- Pythonic (same logic, one-pass dict update) ---
def rank_players_v2(players, results):
    stats = {p: [0, 0] for p in players}   # [wins, losses]
    for winner, loser in results:
        stats[winner][0] += 1
        stats[loser][1]  += 1
    return sorted(players, key=lambda p: (-stats[p][0] * 3, stats[p][1], p))

# Tests
players = ["Alice", "Bob", "Carol", "Dave"]
results = [
    ("Alice","Bob"),("Carol","Dave"),("Bob","Carol"),
    ("Alice","Dave"),("Bob","Dave"),("Alice","Carol"),
]
for fn in (rank_players_v1, rank_players_v2):
    print(fn(players, results))
    # ['Alice', 'Bob', 'Carol', 'Dave']

players2 = ["Zara", "Amy", "Ben"]
results2 = [("Zara","Ben"),("Amy","Zara"),("Amy","Ben")]
for fn in (rank_players_v1, rank_players_v2):
    print(fn(players2, results2))
    # ['Amy', 'Zara', 'Ben']

**Key insight:** A **tuple sort key** `(-points, losses, name)` elegantly encodes all three ranking criteria in one `sorted()` call. Python sorts tuples element by element - so it tries `−points` first, falls back to `losses` on a tie, then `name`. This pattern appears frequently in competitive-style problems.

---
## Run All Tests

In [ ]:
# Run this cell to verify all solutions at once
def test(name, got, expected):
    status = "PASS" if got == expected else "FAIL"
    if status == "FAIL":
        print(f"{status} [{name}]  got={got}  expected={expected}")
    else:
        print(f"{status} [{name}]")

# Q1
test("Q1-a", rearrange_v2([1,2,3,4,5]),   [5,2,3,4,1])
test("Q1-b", rearrange_v2([10,20]),         [20,10])
test("Q1-c", rearrange_v2([7]),             [7])

# Q2
test("Q2-a", clean_summary_v2([3,0,1,0,4,2]),   ([1,2,3,4],10,2.5))
test("Q2-b", clean_summary_v2([0,0,0]),           ([],0,0.0))

# Q3
test("Q3-a", count_pairs_v2([1,5,3,2,4],6),   2)
test("Q3-b", count_pairs_v2([1,1,1,1],2),     6)
test("Q3-c", count_pairs_v2([1,2,3],10),      0)

# Q4
test("Q4-a", categorise_v2([-3,0,2,-1,0,5,4]), {'negative':[-3,-1],'zero':2,'positive':[2,4,5]})
test("Q4-b", categorise_v2([0,0]),              {'negative':[],'zero':2,'positive':[]})

# Q5
test("Q5-a", sort_by_frequency_v1(["the","cat","sat","the","cat","the"]),
     ["the","the","the","cat","cat","sat"])
test("Q5-b", sort_by_frequency_v1(["apple","fig","banana"]),
     ["apple","banana","fig"])

# Q6
test("Q6-a", rotate_90_v2([[1,2,3],[4,5,6],[7,8,9]]), [[7,4,1],[8,5,2],[9,6,3]])
test("Q6-b", rotate_90_v2([[1,2],[3,4]]),              [[3,1],[4,2]])

# Q7
test("Q7-a", saddle_points_v2([[3,1,2],[4,1,0],[2,5,3]]),   [(1,0)])
test("Q7-b", saddle_points_v2([[5,5],[5,5]]),               [(0,0),(0,1),(1,0),(1,1)])

# Q8
test("Q8-enc", run_length_encode_v1([1,1,2,3,3,3,2,2]), [(2,1),(1,2),(3,3),(2,2)])
original = [4,4,4,2,2,1,3,3]
test("Q8-rt",  run_length_decode_v1(run_length_encode_v1(original)), original)

# Q9
test("Q9-a", spiral_boundary_sum_v2([[1,2,3,4],[5,6,7,8],[9,10,11,12]]), 65)
test("Q9-b", spiral_boundary_sum_v2([[1,2,3],[4,5,6],[7,8,9]]),          40)
test("Q9-c", spiral_boundary_sum_v2([[5]]),                               5)

# Q10
players  = ["Alice","Bob","Carol","Dave"]
results  = [("Alice","Bob"),("Carol","Dave"),("Bob","Carol"),("Alice","Dave"),("Bob","Dave"),("Alice","Carol")]
test("Q10-a", rank_players_v1(players, results), ["Alice","Bob","Carol","Dave"])
test("Q10-b", rank_players_v1(["Zara","Amy","Ben"],
     [("Zara","Ben"),("Amy","Zara"),("Amy","Ben")]), ["Amy","Zara","Ben"])